# Insecure HTTP Exposure Scanner with Shodan Validation

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install python-nmap requests
!apt-get install -y nmap


In [ ]:
import nmap
import ipaddress
import requests
import socket
import os
import json
from datetime import datetime

# Optional: Insert your Shodan API key here
SHODAN_API_KEY = 'YOUR_SHODAN_API_KEY'

# Terminal colors (
GREEN = ''
YELLOW = ''
RED = ''
BLUE = ''
RESET = ''

LOCAL_NET = '192.168.1.0/24'  # Adjust this for your local network
REPORT_FILE = 'insecure_http_report.txt'
vulnerable_hosts = []

def scan_network():
    print(f"[*] Scanning network: {LOCAL_NET}")
    nm = nmap.PortScanner()
    try:
        nm.scan(hosts=LOCAL_NET, arguments='-p 80 --open')
    except Exception as e:
        print(f"Nmap scan failed: {e}")
        return []
    return nm

def check_auth(ip):
    try:
        url = f"http://{ip}"
        response = requests.get(url, timeout=3)
        if response.status_code == 401:
            return True
        return False
    except requests.RequestException:
        return None

def shodan_lookup(ip):
    if SHODAN_API_KEY == 'YOUR_SHODAN_API_KEY':
        return None
    try:
        url = f"https://api.shodan.io/shodan/host/{ip}?key={SHODAN_API_KEY}"
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json()
            return True if 'http' in str(data) else False
        return False
    except:
        return None

def generate_report(entries):
    with open(REPORT_FILE, 'w') as f:
        f.write("Insecure HTTP Devices Report\n")
        f.write(f"Generated: {datetime.now()}\n\n")
        for entry in entries:
            f.write(f"{entry['ip']} - HTTP exposed, Auth: {entry['auth']}, Shodan: {entry['shodan']}\n")
    print(f"[✓] Report saved to {REPORT_FILE}")

def main():
    print("🔍 Insecure HTTP Exposure Scanner w/ Shodan Check")
    scanner = scan_network()

    print("\n[!] Checking for insecure HTTP endpoints...\n")
    for host in scanner.all_hosts():
        if scanner[host].has_tcp(80):
            auth_required = check_auth(host)
            if auth_required is None:
                print(f"[?] {host}: No response")
                continue

            shodan_exposed = shodan_lookup(host)

            print(f"[+] {host} - HTTP {'NO AUTH' if not auth_required else 'AUTH'} - Shodan: {'YES' if shodan_exposed else 'NO'}")

            if not auth_required:
                vulnerable_hosts.append({
                    'ip': host,
                    'auth': 'No',
                    'shodan': 'Yes' if shodan_exposed else 'No'
                })

    generate_report(vulnerable_hosts)

main()
